# BigQuery Migration: SQLite → BigQuery | Olist E-Commerce

## Introduction

This notebook documents **syntax differences only** when migrating the Olist e-commerce SQL analysis from SQLite to Google BigQuery.

Each query is paired with a brief explanation of the relevant syntax change. For full business context, findings, and recommendations see [olist_analysis.ipynb](olist_analysis.ipynb).

**Project:** `olist-491115`  
**Dataset:** `olist_ecommerce`

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='olist-491115')

def run(query):
    return client.query(query).to_dataframe()

## Syntax Differences Reference

| # | Difference | SQLite | BigQuery |
|---|-----------|--------|----------|
| 1 | **Table references** | `olist_orders_dataset` | `` `olist-491115.olist_ecommerce.orders` `` |
| 2 | **Date extraction** | `strftime('%Y', col)` | `EXTRACT(YEAR FROM col)` |
| 3 | **String literals** | Double or single quotes allowed | Single quotes only — `'value'` |
| 4 | **Strict GROUP BY** | Non-aggregated columns can be omitted | All non-aggregated SELECT columns must appear in GROUP BY |
| 5 | **Schema auto-detection** | N/A | Auto-detect may misclassify timestamp columns as `STRING`; fix with `TIMESTAMP(col)` or explicit schema on load |

## 1. Order Status Overview

**Syntax difference:** None — this query translates directly with only the table reference updated.

In [ ]:
df = run("""
SELECT order_status, COUNT(order_status) AS num_orders
FROM `olist-491115.olist_ecommerce.orders`
GROUP BY order_status
ORDER BY num_orders DESC
""")
df

## 2. Order Volume by Year

**Syntax difference:** `strftime('%Y', col)` → `EXTRACT(YEAR FROM col)`. BigQuery does not support `strftime`; use the standard SQL `EXTRACT()` function. The GROUP BY must also use the full expression, not the alias.

In [ ]:
df = run("""
SELECT
    EXTRACT(YEAR FROM order_purchase_timestamp) AS year,
    COUNT(*) AS total_orders_placed
FROM `olist-491115.olist_ecommerce.orders`
GROUP BY EXTRACT(YEAR FROM order_purchase_timestamp)
ORDER BY year
""")
df

## 3. Late Deliveries by State

**Syntax difference:** BigQuery requires single quotes for all string literals — double quotes (`"delivered"`) are not valid. Also, table references must use the full project path with backticks.

In [ ]:
df = run("""
SELECT
    c.customer_state,
    COUNT(*) AS total_late_deliveries
FROM `olist-491115.olist_ecommerce.orders` AS o
JOIN `olist-491115.olist_ecommerce.customers` AS c ON o.customer_id = c.customer_id
WHERE order_status = 'delivered'
  AND order_delivered_customer_date > order_estimated_delivery_date
GROUP BY c.customer_state
ORDER BY total_late_deliveries DESC
LIMIT 20
""")
df

## 4. Revenue by Payment Type

**Syntax difference:** JOIN syntax is identical, but BigQuery table references require the full `project.dataset.table` path wrapped in backticks. Single quotes are enforced for string literals throughout.

In [ ]:
df = run("""
SELECT payment_type, SUM(payment_value) AS total_value
FROM `olist-491115.olist_ecommerce.orders` AS o
JOIN `olist-491115.olist_ecommerce.order_payments` AS op ON o.order_id = op.order_id
GROUP BY payment_type
ORDER BY total_value DESC
""")
df

## 5. Top Product Categories by Revenue

**Syntax difference:** Four-table JOIN structure is unchanged, but every table reference must use the full BigQuery path. SQLite's short table names (`olist_products_dataset`, `product_category_name_translation`) become qualified paths in `olist_ecommerce`.

In [ ]:
df = run("""
SELECT t.product_category_name_english, SUM(op.payment_value) AS total_value
FROM `olist-491115.olist_ecommerce.order_items` AS oi
JOIN `olist-491115.olist_ecommerce.products` AS p ON oi.product_id = p.product_id
JOIN `olist-491115.olist_ecommerce.order_payments` AS op ON oi.order_id = op.order_id
JOIN `olist-491115.olist_ecommerce.category_translation` AS t ON p.product_category_name = t.product_category_name
GROUP BY t.product_category_name_english
ORDER BY total_value DESC
LIMIT 10
""")
df

## 6. Seller Rankings

**Syntax difference:** BigQuery enforces strict GROUP BY — `seller_state` is non-aggregated and must appear explicitly in GROUP BY. SQLite silently accepts `GROUP BY seller_id` alone and picks an arbitrary value for `seller_state`; BigQuery raises an error.

In [ ]:
df = run("""
SELECT
    oi.seller_id,
    SUM(op.payment_value) AS total_payment_value,
    DENSE_RANK() OVER (ORDER BY SUM(op.payment_value) DESC) AS ranking,
    s.seller_state
FROM `olist-491115.olist_ecommerce.order_items` AS oi
JOIN `olist-491115.olist_ecommerce.order_payments` AS op ON oi.order_id = op.order_id
JOIN `olist-491115.olist_ecommerce.sellers` AS s ON oi.seller_id = s.seller_id
GROUP BY oi.seller_id, s.seller_state
ORDER BY total_payment_value DESC
LIMIT 20
""")
df

## 7. Average Order Value by Month

**Syntax difference:** CTE syntax is identical in both SQLite and BigQuery.
The key change is `strftime('%m', col)` becomes `EXTRACT(MONTH FROM col)`.
Note: Unlike SQLite, BigQuery does NOT support `strftime` at all — 
`EXTRACT()` is the standard SQL approach and works across all major 
cloud databases including BigQuery, PostgreSQL and Snowflake.

In [ ]:
df = run("""
WITH order_payments AS (
    SELECT order_id, payment_value
    FROM `olist-491115.olist_ecommerce.order_payments`
),
order_month AS (
    SELECT order_id, order_purchase_timestamp
    FROM `olist-491115.olist_ecommerce.orders`
)
SELECT
    AVG(a.payment_value) AS average_payment_value,
    EXTRACT(MONTH FROM b.order_purchase_timestamp) AS month
FROM order_payments AS a
JOIN order_month AS b ON a.order_id = b.order_id
GROUP BY EXTRACT(MONTH FROM b.order_purchase_timestamp)
ORDER BY average_payment_value DESC
""")
df

## Summary
All 7 queries migrated successfully from SQLite to BigQuery.
**Key takeaway:** BigQuery is stricter and more explicit than SQLite,
but the core SQL logic remains identical.
